# SAJU ML V2 External DEV Diagnostics R2

**Notebook version: `V2_DEV_DIAGNOSTICS_R2_20260815`**

This replaces the earlier diagnostics notebook.

The earlier notebook incorrectly loaded only the V1 base feature cache.
That cache contains TENGOD features but **not** the SHINSAL features introduced in V1.3.

R2 explicitly loads:

```text
research/ml/artifacts/v1/ml_feature_table_v1.parquet or .csv
research/ml/artifacts/v1_3/extended_astrology_features_v1_3.csv
```

and merges them before fitting the frozen V1 candidates.

## Hard boundaries

- diagnostics only
- V2 NEW_DEV only
- NEW_CONFIRM is never referenced or loaded
- no feature tuning
- no hyperparameter tuning
- no new candidate selection
- frozen Pairwise Logistic Regression, L2, C=0.3

In [1]:
from pathlib import Path
from itertools import product
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "V2_DEV_DIAGNOSTICS_R2_20260815"
SEED = 20260815
N_PERMUTATIONS = 2000
N_BOOTSTRAP = 20000

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run from inside the saju repo.")

REPO_ROOT = find_repo_root()

V1_CORPUS_PATH = REPO_ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json"
V1_BASE_DIR = REPO_ROOT / "research/ml/artifacts/v1"
V13_EXTENDED_PATH = (
    REPO_ROOT
    / "research/ml/artifacts/v1_3/extended_astrology_features_v1_3.csv"
)

V2_DEV_PATH = (
    REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json"
)
V2_FEATURE_PATH = (
    REPO_ROOT / "research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv"
)
V2_RESULTS_PATH = (
    REPO_ROOT / "research/ml/artifacts/v2/V2_DEV_candidate_results.csv"
)
V2_FREEZE_PATH = (
    REPO_ROOT / "research/ml/artifacts/v2/V2_DEV_WINNER_FREEZE.json"
)
CANDIDATE_FREEZE_PATH = (
    REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_V2_CANDIDATE_FREEZE.json"
)

OUT = REPO_ROOT / "research/ml/artifacts/v2_diagnostics_R2"
OUT.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("NOTEBOOK VERSION:", NOTEBOOK_VERSION)
print("repo:", REPO_ROOT)
print("output:", OUT)
print("=" * 70)

NOTEBOOK VERSION: V2_DEV_DIAGNOSTICS_R2_20260815
repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2_diagnostics_R2


## 1. File preflight

This cell must print `ALL REQUIRED FILES FOUND`.

If the V1.3 extended cache is absent, rerun:

```text
research/ml/saju_ml_feature_sweep_v1_3.ipynb
```

before continuing.

In [2]:
required = {
    "V1 corpus": V1_CORPUS_PATH,
    "V1.3 extended astrology features": V13_EXTENDED_PATH,
    "V2 NEW_DEV corpus": V2_DEV_PATH,
    "V2 NEW_DEV feature cache": V2_FEATURE_PATH,
    "V2 candidate results": V2_RESULTS_PATH,
    "V2 winner freeze": V2_FREEZE_PATH,
    "candidate freeze": CANDIDATE_FREEZE_PATH,
}

missing_files = []

for label, path in required.items():
    ok = path.exists()
    print("%-36s %s  %s" % (label, "OK" if ok else "MISSING", path))
    if not ok:
        missing_files.append((label, str(path)))

v1_parquet = V1_BASE_DIR / "ml_feature_table_v1.parquet"
v1_csv = V1_BASE_DIR / "ml_feature_table_v1.csv"

if not v1_parquet.exists() and not v1_csv.exists():
    missing_files.append(
        ("V1 base feature cache", str(v1_parquet) + " OR " + str(v1_csv))
    )

if missing_files:
    raise FileNotFoundError(
        "Required files missing: %s" % missing_files
    )

print("\nALL REQUIRED FILES FOUND")

V1 corpus                            OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json
V1.3 extended astrology features     OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v1_3/extended_astrology_features_v1_3.csv
V2 NEW_DEV corpus                    OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json
V2 NEW_DEV feature cache             OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2/V2_NEW_DEV_feature_table_R2.csv
V2 candidate results                 OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2/V2_DEV_candidate_results.csv
V2 winner freeze                     OK  /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v2/V2_DEV_WINNER_FREEZE.json
candidate freeze                     OK  /Users/sangjinlee/Desktop/projects/saju/research/ml_corpus/v2/SAJU_ML_V2_CANDIDATE_FREEZE.json

ALL REQUIRED FILES FOUND


## 2. Verify frozen V2 status and ensure confirmation was never loaded

In [3]:
v2_freeze = json.load(open(V2_FREEZE_PATH, encoding="utf-8"))

assert v2_freeze["status"] == "V2_DEV_NO_CANDIDATE_SURVIVES"
assert v2_freeze["winner"] is None
assert v2_freeze["integrity"]["confirm_file_loaded"] is False

dev_results = pd.read_csv(V2_RESULTS_PATH)

print("Frozen status:", v2_freeze["status"])
print("Winner:", v2_freeze["winner"])
print("Confirmation loaded:", v2_freeze["integrity"]["confirm_file_loaded"])
display(dev_results)

Frozen status: V2_DEV_NO_CANDIDATE_SURVIVES
Winner: None
Confirmation loaded: False


,candidate,n_features,macro_pairwise,subject_hit_rate,median_sep,control_pairwise,delta_vs_control,bootstrap_mean_delta,bootstrap_ci_low,bootstrap_ci_high,bootstrap_p_delta_gt_0,fresh_v2_pairwise,prior_unused_candidate_pairwise,high_conf_pairwise
0,TENGOD,10,0.585714,0.600000,0.189361,0.519643,0.066071,0.066071,-0.058929,0.189286,0.8467,0.613636,0.580508,0.588235
1,SHINSAL_ALL,43,0.564286,0.528571,0.443621,0.519643,0.044643,0.044643,-0.083929,0.169643,0.7465,0.613636,0.555085,0.558824


## 3. Build identical Career subject-year labels

Unknown labels are excluded.
Positive/negative collisions in the same subject-year are excluded.
Only subjects with both positive and negative Career years remain pair-evaluable.

In [4]:
def build_target_rows(corpus):
    rows = []

    for subject in corpus["subjects"]:
        birth = subject.get("birth") or {}
        if not birth.get("birth_time_known", True):
            continue

        for event in subject.get("events", []):
            if not event.get("ml_usable", True):
                continue

            label = (event.get("targets") or {}).get("Career", "unknown")
            if label not in ("positive", "negative"):
                continue

            rows.append({
                "subject_id": subject["subject_id"],
                "name": subject["name"],
                "origin_group": subject.get(
                    "origin_group",
                    subject.get("source_pool", "UNKNOWN"),
                ),
                "year": int(event["year"]),
                "label": 1 if label == "positive" else 0,
                "confidence": event.get(
                    "source_event_confidence", "unknown"
                ),
            })

    raw = pd.DataFrame(rows)
    collapsed = []

    confidence_rank = {
        "high": 3,
        "medium": 2,
        "low": 1,
        "unknown": 0,
    }

    for (sid, year), group in raw.groupby(["subject_id", "year"]):
        if group["label"].nunique() > 1:
            continue

        best_confidence = sorted(
            group["confidence"].astype(str),
            key=lambda x: confidence_rank.get(x, 0),
            reverse=True,
        )[0]

        collapsed.append({
            "subject_id": sid,
            "name": group["name"].iloc[0],
            "origin_group": group["origin_group"].iloc[0],
            "year": int(year),
            "label": int(group["label"].iloc[0]),
            "confidence": best_confidence,
        })

    out = pd.DataFrame(collapsed)

    pairable = []
    for sid, group in out.groupby("subject_id"):
        if set(group["label"].tolist()) == set([0, 1]):
            pairable.append(sid)

    return out[out["subject_id"].isin(pairable)].copy()

v1_corpus = json.load(open(V1_CORPUS_PATH, encoding="utf-8"))
v2_corpus = json.load(open(V2_DEV_PATH, encoding="utf-8"))

v1_targets = build_target_rows(v1_corpus)
v2_targets = build_target_rows(v2_corpus)

print("V1 target rows:", len(v1_targets))
print("V1 pairable subjects:", v1_targets["subject_id"].nunique())
print("V2 target rows:", len(v2_targets))
print("V2 pairable subjects:", v2_targets["subject_id"].nunique())

V1 target rows: 113
V1 pairable subjects: 23
V2 target rows: 216
V2 pairable subjects: 70


## 4. Load and merge feature caches

This is the R2 fix.

```text
V1 training rows
= labels
+ V1 base feature cache
+ V1.3 extended astrology feature cache
```

The V1.3 cache contains SHINSAL features.

In [5]:
if v1_parquet.exists():
    v1_base = pd.read_parquet(v1_parquet)
else:
    v1_base = pd.read_csv(v1_csv)

v1_extended = pd.read_csv(V13_EXTENDED_PATH)
v2_features = pd.read_csv(V2_FEATURE_PATH)

print("V1 base rows/cols:", v1_base.shape)
print("V1 extended rows/cols:", v1_extended.shape)
print("V2 feature rows/cols:", v2_features.shape)

v1_df = (
    v1_targets
    .merge(
        v1_base,
        on=["subject_id", "year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        v1_extended,
        on=["subject_id", "year"],
        how="inner",
        validate="one_to_one",
    )
)

v2_df = v2_targets.merge(
    v2_features,
    on=["subject_id", "year"],
    how="inner",
    validate="one_to_one",
)

print("\nMERGE COMPLETE")
print("V1 model rows:", len(v1_df))
print("V1 model subjects:", v1_df["subject_id"].nunique())
print("V2 model rows:", len(v2_df))
print("V2 model subjects:", v2_df["subject_id"].nunique())

v1_shinsal_count = len(
    [c for c in v1_df.columns if c.startswith("shinsal__")]
)
v2_shinsal_count = len(
    [c for c in v2_df.columns if c.startswith("shinsal__")]
)

print("V1 shinsal columns after merge:", v1_shinsal_count)
print("V2 shinsal columns:", v2_shinsal_count)

V1 base rows/cols: (341, 218)
V1 extended rows/cols: (113, 89)
V2 feature rows/cols: (216, 56)

MERGE COMPLETE
V1 model rows: 113
V1 model subjects: 23
V2 model rows: 216
V2 model subjects: 70
V1 shinsal columns after merge: 43
V2 shinsal columns: 43


## 5. Frozen feature preflight

**Do not continue unless this prints:**

```text
FROZEN FEATURE PREFLIGHT PASSED
TENGOD: 10 / 10
SHINSAL: 43 / 43
```

In [6]:
candidate_freeze = json.load(
    open(CANDIDATE_FREEZE_PATH, encoding="utf-8")
)

TENGOD_COLS = list(
    candidate_freeze["candidates"]["TENGOD"]["features"]
)
SHINSAL_COLS = list(
    candidate_freeze["candidates"]["SHINSAL_ALL"]["features"]
)

def missing_columns(frame, columns):
    return [c for c in columns if c not in frame.columns]

checks = []

for frame_name, frame in [
    ("V1_TRAIN", v1_df),
    ("V2_NEW_DEV", v2_df),
]:
    missing_tg = missing_columns(frame, TENGOD_COLS)
    missing_sh = missing_columns(frame, SHINSAL_COLS)

    print("\n", frame_name)
    print(
        "TENGOD: %d / %d present"
        % (len(TENGOD_COLS) - len(missing_tg), len(TENGOD_COLS))
    )
    print(
        "SHINSAL: %d / %d present"
        % (len(SHINSAL_COLS) - len(missing_sh), len(SHINSAL_COLS))
    )

    if missing_tg:
        print("Missing TENGOD:", missing_tg)

    if missing_sh:
        print("Missing SHINSAL first 10:", missing_sh[:10])

    checks.append(
        (frame_name, len(missing_tg), len(missing_sh))
    )

failed = [x for x in checks if x[1] > 0 or x[2] > 0]

if failed:
    raise RuntimeError(
        "Frozen feature preflight FAILED: %s. "
        "Do not run model fitting." % (failed,)
    )

print("\nFROZEN FEATURE PREFLIGHT PASSED")
print("TENGOD: %d / %d" % (len(TENGOD_COLS), len(TENGOD_COLS)))
print("SHINSAL: %d / %d" % (len(SHINSAL_COLS), len(SHINSAL_COLS)))


 V1_TRAIN
TENGOD: 10 / 10 present
SHINSAL: 43 / 43 present

 V2_NEW_DEV
TENGOD: 10 / 10 present
SHINSAL: 43 / 43 present

FROZEN FEATURE PREFLIGHT PASSED
TENGOD: 10 / 10
SHINSAL: 43 / 43


## 6. Fit frozen V1 candidates and score untouched V2 NEW_DEV

In [7]:
MODEL = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        penalty="l2",
        C=0.3,
        max_iter=3000,
        random_state=SEED,
    )),
])

def make_pairwise_training(frame, columns):
    X_rows = []
    y_rows = []

    for sid, group in frame.groupby("subject_id"):
        positive = group[group["label"] == 1]
        negative = group[group["label"] == 0]

        for _, p_row in positive.iterrows():
            for _, n_row in negative.iterrows():
                diff = (
                    p_row[columns].to_numpy(dtype=float)
                    - n_row[columns].to_numpy(dtype=float)
                )

                X_rows.append(diff)
                y_rows.append(1)

                X_rows.append(-diff)
                y_rows.append(0)

    return np.asarray(X_rows), np.asarray(y_rows)

def fit_frozen(frame, columns):
    X, y = make_pairwise_training(frame, columns)

    model = clone(MODEL)
    model.fit(X, y)
    return model

def score_frozen(model, frame, columns):
    scaler = model.named_steps["scale"]
    classifier = model.named_steps["model"]

    X = frame[columns].to_numpy(dtype=float)
    X_scaled = scaler.transform(X)

    return X_scaled.dot(classifier.coef_.ravel())

def subject_pairwise(frame, score_col):
    rows = []

    for sid, group in frame.groupby("subject_id"):
        positive = group.loc[
            group["label"] == 1, score_col
        ].dropna().to_numpy(dtype=float)

        negative = group.loc[
            group["label"] == 0, score_col
        ].dropna().to_numpy(dtype=float)

        if len(positive) == 0 or len(negative) == 0:
            continue

        comparisons = []

        for p_score, n_score in product(positive, negative):
            if p_score > n_score:
                comparisons.append(1.0)
            elif p_score == n_score:
                comparisons.append(0.5)
            else:
                comparisons.append(0.0)

        rows.append({
            "subject_id": sid,
            "pairwise": float(np.mean(comparisons)),
            "hit": float(
                np.mean(positive) > np.mean(negative)
            ),
            "sep": float(
                np.mean(positive) - np.mean(negative)
            ),
        })

    return pd.DataFrame(rows)

models = {
    "TENGOD": fit_frozen(v1_df, TENGOD_COLS),
    "SHINSAL_ALL": fit_frozen(v1_df, SHINSAL_COLS),
}

scored = v2_df[
    [
        "subject_id",
        "name",
        "origin_group",
        "year",
        "label",
        "confidence",
        "baseline__control_score",
    ]
].copy()

scored["TENGOD"] = score_frozen(
    models["TENGOD"], v2_df, TENGOD_COLS
)
scored["SHINSAL_ALL"] = score_frozen(
    models["SHINSAL_ALL"], v2_df, SHINSAL_COLS
)

print("Frozen candidate scoring complete.")
print("Rows:", len(scored))
print("Subjects:", scored["subject_id"].nunique())

scored.to_csv(
    OUT / "V2_DEV_scored_rows_diagnostic_R2.csv",
    index=False,
)

Frozen candidate scoring complete.
Rows: 216
Subjects: 70


## 7. Fixed-model permutation test on external NEW_DEV

In [8]:
def macro_pairwise(frame, score_col):
    subject_df = subject_pairwise(frame, score_col)
    return float(subject_df["pairwise"].mean())

def fixed_model_permutation(frame, score_col, n_perm):
    observed = macro_pairwise(frame, score_col)
    rng = np.random.RandomState(
        SEED + sum([ord(x) for x in score_col])
    )

    null_scores = []

    for _ in range(n_perm):
        perm = frame.copy()

        for sid, idx in perm.groupby("subject_id").groups.items():
            idx = list(idx)
            labels = perm.loc[idx, "label"].to_numpy().copy()
            rng.shuffle(labels)
            perm.loc[idx, "label"] = labels

        null_scores.append(
            macro_pairwise(perm, score_col)
        )

    null_scores = np.asarray(null_scores)

    p_value = (
        1.0 + float(np.sum(null_scores >= observed))
    ) / (len(null_scores) + 1.0)

    return {
        "candidate": score_col,
        "observed": observed,
        "null_mean": float(np.mean(null_scores)),
        "null_p95": float(np.quantile(null_scores, 0.95)),
        "p_value": p_value,
    }

permutation_rows = []

for candidate in ["TENGOD", "SHINSAL_ALL"]:
    permutation_rows.append(
        fixed_model_permutation(
            scored,
            candidate,
            N_PERMUTATIONS,
        )
    )

permutation_df = pd.DataFrame(permutation_rows)
display(permutation_df)

permutation_df.to_csv(
    OUT / "V2_DEV_fixed_model_permutation_R2.csv",
    index=False,
)

,candidate,observed,null_mean,null_p95,p_value
0,TENGOD,0.585714,0.499242,0.576786,0.034483
1,SHINSAL_ALL,0.564286,0.500841,0.582143,0.102449


## 8. Bootstrap vs chance and Production Control

In [9]:
control_subjects = subject_pairwise(
    scored,
    "baseline__control_score",
)

bootstrap_rows = []

for candidate in ["TENGOD", "SHINSAL_ALL"]:
    candidate_subjects = subject_pairwise(
        scored,
        candidate,
    )

    vals = candidate_subjects["pairwise"].to_numpy(dtype=float)
    rng = np.random.RandomState(SEED + len(candidate))

    chance_boot = np.asarray([
        rng.choice(
            vals,
            size=len(vals),
            replace=True,
        ).mean()
        for _ in range(N_BOOTSTRAP)
    ])

    paired = (
        candidate_subjects[
            ["subject_id", "pairwise"]
        ]
        .rename(columns={"pairwise": "candidate_pairwise"})
        .merge(
            control_subjects[
                ["subject_id", "pairwise"]
            ].rename(
                columns={"pairwise": "control_pairwise"}
            ),
            on="subject_id",
            how="inner",
        )
    )

    paired["delta"] = (
        paired["candidate_pairwise"]
        - paired["control_pairwise"]
    )

    delta = paired["delta"].to_numpy(dtype=float)

    delta_boot = np.asarray([
        rng.choice(
            delta,
            size=len(delta),
            replace=True,
        ).mean()
        for _ in range(N_BOOTSTRAP)
    ])

    bootstrap_rows.append({
        "candidate": candidate,
        "macro_pairwise": float(vals.mean()),
        "pairwise_ci_low": float(np.quantile(chance_boot, 0.025)),
        "pairwise_ci_high": float(np.quantile(chance_boot, 0.975)),
        "bootstrap_p_pairwise_gt_0_5": float(
            np.mean(chance_boot > 0.5)
        ),
        "delta_vs_control": float(delta.mean()),
        "delta_ci_low": float(np.quantile(delta_boot, 0.025)),
        "delta_ci_high": float(np.quantile(delta_boot, 0.975)),
        "bootstrap_p_delta_gt_0": float(
            np.mean(delta_boot > 0)
        ),
        "subjects_better": int(np.sum(delta > 0)),
        "subjects_equal": int(np.sum(delta == 0)),
        "subjects_worse": int(np.sum(delta < 0)),
    })

    paired.to_csv(
        OUT / (
            candidate
            + "_subject_deltas_diagnostic_R2.csv"
        ),
        index=False,
    )

bootstrap_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_df)

bootstrap_df.to_csv(
    OUT / "V2_DEV_bootstrap_diagnostic_R2.csv",
    index=False,
)

,candidate,macro_pairwise,pairwise_ci_low,pairwise_ci_high,bootstrap_p_pairwise_gt_0_5,delta_vs_control,delta_ci_low,delta_ci_high,bootstrap_p_delta_gt_0,subjects_better,subjects_equal,subjects_worse
0,TENGOD,0.585714,0.492857,0.675000,0.96175,0.066071,-0.057143,0.189286,0.84625,29,19,22
1,SHINSAL_ALL,0.564286,0.471429,0.657143,0.90625,0.044643,-0.082143,0.173214,0.75060,26,21,23


## 9. Origin-group and high-confidence sensitivity

In [10]:
group_rows = []

for score_col in [
    "TENGOD",
    "SHINSAL_ALL",
    "baseline__control_score",
]:
    for origin, group in scored.groupby("origin_group"):
        result = subject_pairwise(group, score_col)

        if len(result) == 0:
            continue

        group_rows.append({
            "score": score_col,
            "origin_group": origin,
            "n_subjects": len(result),
            "macro_pairwise": float(
                result["pairwise"].mean()
            ),
            "hit_rate": float(
                result["hit"].mean()
            ),
        })

origin_df = pd.DataFrame(group_rows)
display(origin_df)

origin_df.to_csv(
    OUT / "V2_DEV_origin_group_diagnostic_R2.csv",
    index=False,
)

high = scored[scored["confidence"] == "high"].copy()

high_rows = []

for score_col in [
    "TENGOD",
    "SHINSAL_ALL",
    "baseline__control_score",
]:
    result = subject_pairwise(high, score_col)

    if len(result) == 0:
        continue

    high_rows.append({
        "score": score_col,
        "n_subjects": len(result),
        "macro_pairwise": float(
            result["pairwise"].mean()
        ),
        "hit_rate": float(
            result["hit"].mean()
        ),
    })

high_df = pd.DataFrame(high_rows)
display(high_df)

high_df.to_csv(
    OUT / "V2_DEV_high_conf_diagnostic_R2.csv",
    index=False,
)

,score,origin_group,n_subjects,macro_pairwise,hit_rate
0,TENGOD,FRESH_V2_AA,11,0.613636,0.636364
1,TENGOD,PRIOR_UNUSED_PUBLIC_CANDIDATE,59,0.580508,0.593220
2,SHINSAL_ALL,FRESH_V2_AA,11,0.613636,0.727273
3,SHINSAL_ALL,PRIOR_UNUSED_PUBLIC_CANDIDATE,59,0.555085,0.491525
4,baseline__control_score,FRESH_V2_AA,11,0.511364,0.363636
5,baseline__control_score,PRIOR_UNUSED_PUBLIC_CANDIDATE,59,0.521186,0.457627


,score,n_subjects,macro_pairwise,hit_rate
0,TENGOD,17,0.588235,0.588235
1,SHINSAL_ALL,17,0.558824,0.411765
2,baseline__control_score,17,0.588235,0.529412


## 10. Ten-God distribution drift V1 vs V2

In [11]:
drift_rows = []

for feature in TENGOD_COLS:
    v1_values = pd.to_numeric(
        v1_df[feature],
        errors="coerce",
    ).fillna(0.0)

    v2_values = pd.to_numeric(
        v2_df[feature],
        errors="coerce",
    ).fillna(0.0)

    drift_rows.append({
        "feature": feature,
        "v1_mean": float(v1_values.mean()),
        "v2_mean": float(v2_values.mean()),
        "abs_mean_shift": float(
            abs(v1_values.mean() - v2_values.mean())
        ),
        "v1_nonzero_share": float(
            np.mean(v1_values != 0)
        ),
        "v2_nonzero_share": float(
            np.mean(v2_values != 0)
        ),
    })

drift_df = pd.DataFrame(drift_rows).sort_values(
    "abs_mean_shift",
    ascending=False,
)

display(drift_df)

drift_df.to_csv(
    OUT / "V2_DEV_tengod_feature_drift_R2.csv",
    index=False,
)

,feature,v1_mean,v2_mean,abs_mean_shift,v1_nonzero_share,v2_nonzero_share
1,tengod__dw_group_output,0.371681,0.180556,0.191126,0.371681,0.180556
3,tengod__dw_group_officer,0.070796,0.254630,0.183833,0.070796,0.254630
8,tengod__sw_group_officer,0.168142,0.250000,0.081858,0.168142,0.250000
4,tengod__dw_group_resource,0.159292,0.217593,0.058301,0.159292,0.217593
5,tengod__sw_group_peer,0.256637,0.199074,0.057563,0.256637,0.199074
2,tengod__dw_group_wealth,0.212389,0.166667,0.045723,0.212389,0.166667
7,tengod__sw_group_wealth,0.203540,0.180556,0.022984,0.203540,0.180556
0,tengod__dw_group_peer,0.185841,0.180556,0.005285,0.185841,0.180556
9,tengod__sw_group_resource,0.194690,0.189815,0.004875,0.194690,0.189815
6,tengod__sw_group_output,0.176991,0.180556,0.003564,0.176991,0.180556


## 11. Diagnostic decision

This **cannot** reverse the frozen result:

```text
V2_DEV_NO_CANDIDATE_SURVIVES
```

The only possible diagnostic recommendations are:

- `COLLECT_MORE_FRESH_DEV_AROUND_TENGOD`
- `RETHINK_BOTH_CANDIDATES`

NEW_CONFIRM remains sealed either way.

In [12]:
tg_result = dev_results[
    dev_results["candidate"] == "TENGOD"
].iloc[0]

sh_result = dev_results[
    dev_results["candidate"] == "SHINSAL_ALL"
].iloc[0]

tg_perm = permutation_df[
    permutation_df["candidate"] == "TENGOD"
].iloc[0]

if (
    tg_result["macro_pairwise"] > 0.55
    and tg_perm["p_value"] < 0.10
    and tg_result["macro_pairwise"]
        > sh_result["macro_pairwise"]
):
    diagnostic_status = (
        "COLLECT_MORE_FRESH_DEV_AROUND_TENGOD"
    )
else:
    diagnostic_status = "RETHINK_BOTH_CANDIDATES"

decision = {
    "notebook_version": NOTEBOOK_VERSION,
    "frozen_v2_status": v2_freeze["status"],
    "diagnostic_status": diagnostic_status,
    "confirmation_must_remain_sealed": True,
    "confirmation_loaded": False,
    "note": (
        "Diagnostic result does not modify the predeclared "
        "V2 gate. No retuning or confirmation scoring."
    ),
}

print(json.dumps(decision, indent=2))

with open(
    OUT / "V2_DEV_DIAGNOSTIC_DECISION_R2.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(decision, f, indent=2)

{
  "notebook_version": "V2_DEV_DIAGNOSTICS_R2_20260815",
  "frozen_v2_status": "V2_DEV_NO_CANDIDATE_SURVIVES",
  "diagnostic_status": "COLLECT_MORE_FRESH_DEV_AROUND_TENGOD",
  "confirmation_must_remain_sealed": true,
  "confirmation_loaded": false,
  "note": "Diagnostic result does not modify the predeclared V2 gate. No retuning or confirmation scoring."
}


## Send these files back

```text
research/ml/artifacts/v2_diagnostics_R2/
├── V2_DEV_fixed_model_permutation_R2.csv
├── V2_DEV_bootstrap_diagnostic_R2.csv
├── V2_DEV_origin_group_diagnostic_R2.csv
├── V2_DEV_high_conf_diagnostic_R2.csv
├── V2_DEV_tengod_feature_drift_R2.csv
└── V2_DEV_DIAGNOSTIC_DECISION_R2.json
```

Before trusting the run, confirm the notebook printed:

```text
NOTEBOOK VERSION: V2_DEV_DIAGNOSTICS_R2_20260815
FROZEN FEATURE PREFLIGHT PASSED
TENGOD: 10 / 10
SHINSAL: 43 / 43
```